# 🚀 Ultimate Google Drive Model Hub & Downloader
### Download open-source models + Save & Load models built from scratch directly into your 5TB Google Drive.

---

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
import os
import torch

drive.mount('/content/drive')

# Set base model directory in your Google Drive (5TB)
BASE_DRIVE_DIR = "/content/drive/MyDrive/LLM_Model_Hub"
OPENSOURCE_DIR = os.path.join(BASE_DRIVE_DIR, "open_source_models")
SCRATCH_DIR = os.path.join(BASE_DRIVE_DIR, "from_scratch_models")

os.makedirs(OPENSOURCE_DIR, exist_ok=True)
os.makedirs(SCRATCH_DIR, exist_ok=True)

print(f"✅ Google Drive Mounted!")
print(f"📁 Open-Source Models Folder: {OPENSOURCE_DIR}")
print(f"📁 From-Scratch Models Folder: {SCRATCH_DIR}")

In [ ]:
# Step 2: Install required downloading and conversion libraries
!pip install -q huggingface_hub safetensors torch tqdm

## 📥 Part 1: Download Open-Source Models to Drive
Select from curated models (AWQ/4-bit for vLLM or GGUF) or download any custom HuggingFace repo.

In [ ]:
from huggingface_hub import snapshot_download

# Optional: Set your HuggingFace token if downloading gated models (e.g. Meta-Llama)
HF_TOKEN = ""  # Paste HF token here if needed: https://huggingface.co/settings/tokens

CURATED_MODELS = {
    "1": ("Llama-3-8B-Instruct-AWQ (Best for Colab vLLM)", "casperhansen/llama-3-8b-instruct-awq"),
    "2": ("Qwen-2.5-Coder-7B-Instruct-AWQ (Top Coding Model)", "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"),
    "3": ("DeepSeek-R1-Distill-Qwen-7B-AWQ (Reasoning Model)", "casperhansen/deepseek-r1-distill-qwen-7b-awq"),
    "4": ("Mistral-7B-Instruct-v0.3-AWQ (Fast General Model)", "casperhansen/mistral-7b-instruct-v0.3-awq"),
    "5": ("TinyLlama-1.1B (Ultra lightweight draft model)", "TinyLlama/TinyLlama-1.1B-Chat-v1.0"),
    "6": ("SmolLM2-1.7B-Instruct (High quality small model)", "HuggingFaceTB/SmolLM2-1.7B-Instruct")
}

def download_hf_model(repo_id, custom_folder_name=None):
    folder_name = custom_folder_name or repo_id.replace("/", "--")
    dest_dir = os.path.join(OPENSOURCE_DIR, folder_name)
    print(f"\n⏳ Starting download for: {repo_id}")
    print(f"💾 Target Drive location: {dest_dir}")
    
    snapshot_download(
        repo_id=repo_id,
        local_dir=dest_dir,
        local_dir_use_symlinks=False,
        token=HF_TOKEN if HF_TOKEN else None
    )
    print(f"\n🎉 Successfully saved {repo_id} to Google Drive!")
    return dest_dir

# Display menu
print("=== Curated Open-Source Models ===")
for key, (desc, repo) in CURATED_MODELS.items():
    print(f"[{key}] {desc} -> {repo}")
print("[custom] Or enter any Hugging Face repo ID (e.g. 'deepseek-ai/DeepSeek-V2-Lite')")

In [ ]:
# Run this cell to download a model (Change MODEL_CHOICE to your desired option)
MODEL_CHOICE = "1"  # '1', '2', '3', '4', '5', '6' or enter custom repo ID like 'meta-llama/Llama-3.2-1B'

if MODEL_CHOICE in CURATED_MODELS:
    repo_id = CURATED_MODELS[MODEL_CHOICE][1]
    download_hf_model(repo_id)
else:
    download_hf_model(MODEL_CHOICE)

## 🛠️ Part 2: Save & Load From-Scratch Models to Drive
Use these helper functions when training or building your own models (e.g., DeepSeek, GPT-2, NanoGPT, custom Transformers).

In [ ]:
import json
from datetime import datetime

class ScratchModelManager:
    def __init__(self, base_dir=SCRATCH_DIR):
        self.base_dir = base_dir

    def save_model(self, model, model_name, config=None, optimizer=None, step=None, loss=None):
        """
        Saves a PyTorch model built from scratch directly into Google Drive.
        """
        save_path = os.path.join(self.base_dir, model_name)
        os.makedirs(save_path, exist_ok=True)
        
        # 1. Save Weights (State Dict)
        weights_file = os.path.join(save_path, "model.pt")
        torch.save(model.state_dict(), weights_file)
        
        # 2. Save Metadata & Config
        meta = {
            "model_name": model_name,
            "saved_at": datetime.utcnow().isoformat(),
            "step": step,
            "loss": loss,
            "config": config or {}
        }
        with open(os.path.join(save_path, "config.json"), "w") as f:
            json.dump(meta, f, indent=2)
            
        # 3. Optional: Checkpoint for resume training
        if optimizer is not None:
            ckpt_file = os.path.join(save_path, "checkpoint.pt")
            torch.save({
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "step": step,
                "loss": loss
            }, ckpt_file)
            
        print(f"💾 Model successfully saved to: {save_path}")
        print(f"   - Weights: {weights_file}")
        print(f"   - Config: {os.path.join(save_path, 'config.json')}")
        return save_path

    def load_model(self, model, model_name, device="cuda" if torch.cuda.is_available() else "cpu"):
        """
        Loads saved weights from Google Drive into a PyTorch model instance.
        """
        weights_file = os.path.join(self.base_dir, model_name, "model.pt")
        if not os.path.exists(weights_file):
            raise FileNotFoundError(f"Weights file not found at: {weights_file}")
            
        state_dict = torch.load(weights_file, map_location=device)
        model.load_state_dict(state_dict)
        model.to(device)
        print(f"✅ Successfully loaded weights for '{model_name}' on {device}")
        return model

    def list_saved_models(self):
        """
        Lists all from-scratch models currently stored in your Drive.
        """
        models = [d for d in os.listdir(self.base_dir) if os.path.isdir(os.path.join(self.base_dir, d))]
        print(f"\n📁 Found {len(models)} from-scratch model(s) in Drive:")
        for m in models:
            config_path = os.path.join(self.base_dir, m, "config.json")
            desc = ""
            if os.path.exists(config_path):
                with open(config_path) as f:
                    data = json.load(f)
                    desc = f"(Saved: {data.get('saved_at', 'N/A')}, Step: {data.get('step', 'N/A')})"
            print(f"  • {m} {desc}")
        return models

manager = ScratchModelManager()
print("✅ ScratchModelManager initialized!")

### Example: Test saving and loading a dummy from-scratch model

In [ ]:
import torch.nn as nn

# Example dummy model from scratch
class SimpleTransformerModel(nn.Module):
    def __init__(self, vocab_size=50257, d_model=256):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
    def forward(self, x):
        return self.lm_head(self.embed(x))

# 1. Instantiate model
my_model = SimpleTransformerModel()

# 2. Save it to Google Drive
config = {"vocab_size": 50257, "d_model": 256, "architecture": "SimpleTransformer"}
manager.save_model(my_model, model_name="my_deepseek_scratch_v1", config=config, step=1000, loss=2.34)

# 3. List all models in Drive
manager.list_saved_models()

# 4. Reload it from Drive into a fresh instance
fresh_model = SimpleTransformerModel()
fresh_model = manager.load_model(fresh_model, model_name="my_deepseek_scratch_v1")

## 📊 Part 3: Drive Storage & Model Inventory Inspector

In [ ]:
import shutil

def get_dir_size_gb(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            if not os.path.islink(fp):
                total += os.path.getsize(fp)
    return total / (1024 ** 3)

print("=== Google Drive Model Inventory ===")
print(f"Root: {BASE_DRIVE_DIR}\n")

print("📂 Open-Source Models:")
for item in os.listdir(OPENSOURCE_DIR):
    p = os.path.join(OPENSOURCE_DIR, item)
    if os.path.isdir(p):
        sz = get_dir_size_gb(p)
        print(f"  • {item}: {sz:.2f} GB")

print("\n📂 From-Scratch Models:")
for item in os.listdir(SCRATCH_DIR):
    p = os.path.join(SCRATCH_DIR, item)
    if os.path.isdir(p):
        sz = get_dir_size_gb(p)
        print(f"  • {item}: {sz:.2f} GB")